In [1]:
import pandas as pd
import pickle
import numpy as np
from tqdm import tqdm
import networkx as nx
from collections import defaultdict

# Overlap with StringDB

In [2]:
ppi_network_file = "/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/atomica_net/9606.protein.links.detailed.v12.0.txt" # StringDB
ppi_network = pd.read_csv(ppi_network_file, sep=' ')
ppi_network['protein1'] = ppi_network['protein1'].apply(lambda x: x.split('.')[1])
ppi_network['protein2'] = ppi_network['protein2'].apply(lambda x: x.split('.')[1])
map_to_uniprot_file = "/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/atomica_net/uniprot_to_ENSP_2025_03_20.tsv"
map_to_uniprot = pd.read_csv(map_to_uniprot_file, sep='\t')
map_to_uniprot['To'] = map_to_uniprot['To'].apply(lambda x: x.split('.')[0])
map_to_uniprot = map_to_uniprot.set_index('To')['From'].to_dict()
ppi_network['uniprot1'] = ppi_network['protein1'].apply(lambda x: map_to_uniprot.get(x, None))
ppi_network['uniprot2'] = ppi_network['protein2'].apply(lambda x: map_to_uniprot.get(x, None))
ppi_network.dropna(subset=['uniprot1', 'uniprot2'], inplace=True)
ppi_network = ppi_network.groupby(['uniprot1', 'uniprot2']).agg({
    'combined_score': 'sum',
    'neighborhood': 'sum',
    'fusion': 'sum',
    'cooccurence': 'sum',
    'experimental': 'sum',
    'database': 'sum',
    'textmining': 'sum'
}).reset_index()

In [3]:
ppi_network

,uniprot1,uniprot2,combined_score,neighborhood,fusion,cooccurence,experimental,database,textmining
0,A0A024R1R8,O14737,422,0,0,0,0,0,115
1,A0A024R1R8,O75380,190,0,0,0,0,0,0
2,A0A024R1R8,O75920,386,0,0,0,0,0,0
3,A0A024R1R8,P0DPB6,213,0,0,0,0,0,0
4,A0A024R1R8,P35544,245,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
11213916,X6R8D5,P0C0P6,236,0,0,236,0,0,0
11213917,X6R8D5,P16070,243,0,0,243,0,0,0
11213918,X6R8D5,Q14DG7,209,0,0,209,0,0,0
11213919,X6R8D5,Q8TER5,211,0,0,211,0,0,0


In [3]:
with open("/n/holylabs/LABS/mzitnik_lab/Users/afang/GET/case_studies/network_analysis/modality_graphs_20250308.pkl", "rb") as f:
    modality_graphs = pickle.load(f)

In [7]:
overlap_edges = []
for modality in modality_graphs:
    G = modality_graphs[modality]['graph']
    uniprot_to_node_idx = modality_graphs[modality]['uniprot_to_node_idx']
    node_idx_to_uniprot = {v: k for k, v in uniprot_to_node_idx.items()}
    for edge_type in ['combined_score','neighborhood','fusion','cooccurence','experimental','database','textmining']:
        stringdb_edges = ppi_network[(ppi_network[edge_type] > 0) & ppi_network['uniprot1'].isin(uniprot_to_node_idx) & ppi_network['uniprot2'].isin(uniprot_to_node_idx)][['uniprot1', 'uniprot2']].drop_duplicates()
        stringdb_edges['node1'] = stringdb_edges['uniprot1'].map(uniprot_to_node_idx)
        stringdb_edges['node2'] = stringdb_edges['uniprot2'].map(uniprot_to_node_idx)
        edges = np.minimum(stringdb_edges['node1'], stringdb_edges['node2']), np.maximum(stringdb_edges['node1'], stringdb_edges['node2'])
        edges = set(zip(edges[0], edges[1]))

        num_overlap = 0
        for (i,j) in G.edges():
            if (i,j) in edges or (j,i) in edges:
                num_overlap += 1
        print(f"{modality} {edge_type} {num_overlap} {num_overlap/len(G.edges())*100:3f}% edges in common")
        overlap_edges.append(
            {
                'modality': modality,
                'edge_type': edge_type,
                'num_overlap': num_overlap,
                'percent_overlap': num_overlap/len(G.edges())*100,
            }
        )
overlap_edges = pd.DataFrame(overlap_edges)

protein combined_score 10722 3.900455% edges in common
protein neighborhood 368 0.133871% edges in common
protein fusion 92 0.033468% edges in common
protein cooccurence 901 0.327766% edges in common
protein experimental 4723 1.718136% edges in common
protein database 1349 0.490740% edges in common
protein textmining 9238 3.360605% edges in common
lipid combined_score 5283 6.221369% edges in common
lipid neighborhood 95 0.111874% edges in common
lipid fusion 12 0.014131% edges in common
lipid cooccurence 675 0.794894% edges in common
lipid experimental 1579 1.859463% edges in common
lipid database 586 0.690086% edges in common
lipid textmining 4995 5.882214% edges in common
nucleic_acid combined_score 6387 6.482027% edges in common
nucleic_acid neighborhood 148 0.150202% edges in common
nucleic_acid fusion 31 0.031461% edges in common
nucleic_acid cooccurence 980 0.994581% edges in common
nucleic_acid experimental 3157 3.203970% edges in common
nucleic_acid database 272 0.276047% edges

In [9]:
overlap_edges.pivot(index='modality', columns='edge_type', values='percent_overlap')

edge_type,combined_score,cooccurence,database,experimental,fusion,neighborhood,textmining
modality,,,,,,,
ion,7.216600,1.797350,0.718487,2.934010,0.031731,0.282182,5.991546
ligand,8.165554,1.297952,1.403574,4.073522,0.069689,0.844975,7.720200
lipid,6.221369,0.794894,0.690086,1.859463,0.014131,0.111874,5.882214
nucleic_acid,6.482027,0.994581,0.276047,3.203970,0.031461,0.150202,5.150506
protein,3.900455,0.327766,0.490740,1.718136,0.033468,0.133871,3.360605


In [11]:
overlap_edges.groupby(['edge_type']).agg({'percent_overlap': 'mean'})

,percent_overlap
edge_type,
combined_score,6.397201
cooccurence,1.042509
database,0.715787
experimental,2.757820
fusion,0.036096
neighborhood,0.304621
textmining,5.621014


# Overlap with PPI network
https://www.nature.com/articles/s41467-021-21770-8

In [2]:
ppi_network_file = "/n/holylabs/LABS/mzitnik_lab/Users/afang/GET/case_studies/network_analysis/PPI_network_41467_2021_21770_MOESM5_ESM.csv"
ppi_network = pd.read_csv(ppi_network_file)

In [29]:
ppi_network_edges = ppi_network.apply(lambda row: (row['node_1_name'], row['node_2_name']), axis=1).tolist()
ppi_network_edges = set(ppi_network_edges)
ppi_network_genes = set(ppi_network['node_1_name'].tolist() + ppi_network['node_2_name'].tolist())
len(ppi_network_edges)

387122

In [4]:
with open("/n/holylabs/LABS/mzitnik_lab/Users/afang/GET/case_studies/network_analysis/modality_graphs_20250308.pkl", "rb") as f:
    modality_graphs = pickle.load(f)

In [2]:
uniprot_human_df = pd.read_csv("/n/holylfs06/LABS/mzitnik_lab/Lab/afang/protein_universe/uniprot/uniprotkb_AND_model_organism_9606_2024_12_16.tsv", sep="\t", usecols=['Entry', 'Entry Name', 'Protein names', 'Gene Names'])
uniprot_to_gene = dict(zip(uniprot_human_df['Entry'], uniprot_human_df['Gene Names']))
uniprot_to_protein_name = dict(zip(uniprot_human_df['Entry'], uniprot_human_df['Protein names']))

In [83]:
def get_overlapping_edges(G, modality):
    overlapping_edges = []

    uniprot_to_node_idx = modality_graphs[modality]['uniprot_to_node_idx']
    node_idx_to_uniprot = {v: k for k, v in uniprot_to_node_idx.items()}

    for (i,j) in G.edges():
        uniprot_i = node_idx_to_uniprot[i]
        uniprot_j = node_idx_to_uniprot[j]
        uniprot_i_gene = uniprot_to_gene.get(uniprot_i, None)
        uniprot_j_gene = uniprot_to_gene.get(uniprot_j, None)

        uniprot_i_gene = set(uniprot_i_gene.split()) if uniprot_i_gene and not pd.isna(uniprot_i_gene) else set()
        uniprot_j_gene = set(uniprot_j_gene.split()) if uniprot_j_gene and not pd.isna(uniprot_j_gene) else set()
        uniprot_i_gene = uniprot_i_gene.intersection(ppi_network_genes)
        uniprot_j_gene = uniprot_j_gene.intersection(ppi_network_genes)

        for gene_i in uniprot_i_gene if uniprot_i_gene else []:
            for gene_j in uniprot_j_gene if uniprot_j_gene else []:
                if (gene_i, gene_j) in ppi_network_edges or (gene_j, gene_i) in ppi_network_edges:
                    overlapping_edges.append((modality, i, j, uniprot_i, uniprot_j, gene_i, gene_j))

    overlapping_edges = pd.DataFrame(overlapping_edges, columns=['modality', 'node_i', 'node_j', 'uniprot_i', 'uniprot_j', 'gene_i', 'gene_j'])
    return overlapping_edges

In [95]:
overlapping_edges = []
for modality in modality_graphs:
    graph = modality_graphs[modality]['graph']
    overlapping_edges.append(get_overlapping_edges(graph, modality))
overlapping_edges = pd.concat(overlapping_edges)
overlapping_edges_summary = []
num_overlapping_edges = overlapping_edges['modality'].value_counts()
for modality in modality_graphs:
    graph = modality_graphs[modality]['graph']
    overlapping_edges_summary.append(
        (modality, graph.number_of_edges(), num_overlapping_edges[modality], num_overlapping_edges[modality]/graph.number_of_edges())
    )
overlapping_edges_summary = pd.DataFrame(overlapping_edges_summary, columns=['modality', 'num_edges', 'num_overlapping_edges', 'fraction_overlapping_edges'])
overlapping_edges_summary

,modality,num_edges,num_overlapping_edges,fraction_overlapping_edges
0,protein,274891,981,0.003569
1,lipid,84917,236,0.002779
2,nucleic_acid,98534,637,0.006465
3,ligand,91837,515,0.005608
4,ion,88241,593,0.006720


In [85]:
overlapping_edges_null = defaultdict(list)
for seed in tqdm(range(100), total=100):
    for modality in modality_graphs:
        graph = modality_graphs[modality]['graph'].copy()
        graph = nx.double_edge_swap(graph, nswap=graph.number_of_edges(), max_tries=graph.number_of_edges()*10, seed=seed)
        res = get_overlapping_edges(graph, modality)
        overlapping_edges_null[modality].append(len(res))

100%|██████████| 100/100 [17:20<00:00, 10.41s/it]


In [91]:
for modality, null_distribution in overlapping_edges_null.items():
    test_value = overlapping_edges['modality'].value_counts()[modality]
    p_value = (np.array(null_distribution) <= test_value).sum() / len(null_distribution)
    print(f"{modality}: {p_value}")

protein: 1.0
lipid: 1.0
nucleic_acid: 1.0
ligand: 1.0
ion: 1.0


In [92]:
test_value

593